## Validation Datasets

The goal is to compare station data to ERA5 data and buoy data to GHRSST data to see how much error we are getting in a sample of regions in our models

From there, we can use that to figure out a reasonable "perturbation" for our model, changing 2m Temperature and SST accordingly to account for likely measurement error and use that to run ensemble models.

In [8]:
#import what you need
import pandas as pd #for buoy data
import re
import xarray as xr
import numpy as np
import pandas as pd
import os
from datetime import datetime



GHRSST Data

In [13]:
#Define a function to add the right time coordinate
def preprocess(ds, filename=None):
    import re
    import pandas as pd
    from xarray import Dataset

    # Extract timestamp from filename like 20170323120000
    match = re.match(r'(\d{14})', os.path.basename(filename))
    if not match:
        raise ValueError(f"Could not extract timestamp from filename: {filename}")
    
    timestamp = pd.to_datetime(match.group(1), format='%Y%m%d%H%M%S')

    # Replace the existing time coordinate with correct timestamp
    ds = ds.assign_coords(time=("time", [timestamp]))
    return ds



In [14]:
folder = '../../01Data/GHRSST/'

file_list = sorted([
    os.path.join(folder, f)
    for f in os.listdir(folder)
    if f.endswith('.nc')
])

datasets = []
for file in file_list:
    ds = xr.open_dataset(file)
    ds = preprocess(ds, filename=file)
    datasets.append(ds)

combined = xr.concat(datasets, dim="time")




Get Buoy Data

In [16]:
##create useful function for reading the buoy data
def load_txt_file(filepath):
    # Step 1: Read lines
    with open(filepath, 'r') as f:
        lines = f.readlines()

    # Step 2: Find the first header line (starting with #)
    header_line = next(line for line in lines if line.startswith("#"))
    column_names = header_line.strip().lstrip("#").split()

    # Step 3: Read the data, skipping any header lines
    df = pd.read_csv(filepath, sep='\s+', comment='#', names=column_names, skiprows=2)

    # Step 4: Make sure date/time columns exist in the DataFrame
    date_cols = ['YY', 'MM', 'DD', 'hh', 'mm']
    if all(col in df.columns for col in date_cols):
        df['datetime'] = pd.to_datetime(df[date_cols].rename(
            columns={'YY': 'year', 'MM': 'month', 'DD': 'day', 'hh': 'hour', 'mm': 'minute'}
        ))
        df.drop(date_cols, axis=1, inplace=True)
        df.set_index('datetime', inplace=True)
    else:
        raise ValueError(f"Date/time columns missing in {filepath}")

    return df

In [17]:
##Buoy data

############42019
file_path = "../../01Data/ValidationDatasets/Buoys/42019h2017.txt"
df42019 = load_txt_file(file_path)

############42035
file_path = "../../01Data/ValidationDatasets/Buoys/42035h2017.txt"
df42035 = load_txt_file(file_path)

############42043
# Step 1: Read and prepare headers
file_path = "../../01Data/ValidationDatasets/Buoys/42043h2017.txt"
df42043 = load_txt_file(file_path)

############mgpt2
# Step 1: Read and prepare headers
file_path = "../../01Data/ValidationDatasets/Buoys/mgpt2h2017.txt"
dfmgpt2 = load_txt_file(file_path)

############sgnt2
# Step 1: Read and prepare headers
file_path = "../../01Data/ValidationDatasets/Buoys/sgnt2h2017.txt"
dfsgnt2 = load_txt_file(file_path)

#########Geographic Locations for the buoys
#42019, 42035, 42043, mgpt2, sgnt2
lats = [27.908, 29.235, 28.982, 29.682, 28.771]
lons = [-95.343, -94.41, -94.899, -94.985, -95.617]

#want WTMP, dates are in a column called datetime I think


From previous, atmospheric code, for context

Going to go through the buoys in loose order as listed above
* 42019
* 42035
* 42043
* mgpt2
* sgnt2

Because of how the data are set up, it is a toss up between iterating through GHRSST data versus iterating through the stations. 

In [20]:
sst0323

<class 'netCDF4.Variable'>
int16 analysed_sst(time, lat, lon)
    long_name: analysed sea surface temperature
    standard_name: sea_surface_foundation_temperature
    units: kelvin
    coordinates: lon lat
    _FillValue: -32768
    add_offset: 273.15
    scale_factor: 0.01
    valid_min: -300
    valid_max: 4500
    source: AMSR2-REMSS-L2P-v2.0, AMSRE-REMSS-L2P-v2.0, TMI-REMSS-L2P-v04, GOES<13,16>-OSISAF-L3C-v2.0, SEVIRI-OSISAF-L3C-v2.0, SLSTRA-C3S-L3C-v2.0, ATSR<1,2>-ESACCI-L3U-v2.0, AATSR-ESACCI-L3U-v2.0, AVHRR<06,07,08,09,10,11,12,14,15,16,17,18,19>-ESACCI-L3U-v2.0, AVHRRMTA-ESACCI-L3U-v2.0, GMI-REMSS-L3U-v2.0, VIIRS<NPP,N20>-OSPO-L3U-v2.0
    reference: C.J. Donlon, M. Martin,J.D. Stark, J. Roberts-Jones, E. Fiedler, W. Wimmer. The operational sea surface temperature and sea ice analysis (OSTIA) system. Remote Sensing Environ., 116 (2012), pp. 140-158 http://dx.doi.org/10.1016/j.rse.2010.10.017
    comment:  OSTIA foundation SST
unlimited dimensions: 
current shape = (1, 3600, 72

In [3]:
## Get useful station information for later work

## Prep some other variables for comparisons
FocalTimePts = [datetime.fromisoformat("2017-03-23T00:00:00"), datetime.fromisoformat("2017-04-01T00:00:00"), 
                datetime.fromisoformat("2017-04-03T00:00:00")]

#tolerance of difference in lat/lon from target to what we use for the mean. 
lonTol = 0.13 #unlikely to get an exact match
latTol = 0.13 #unlikely to get an exact match

In [4]:
## Comparisons for Angleton Lake Jackson
# Find subset within the tolerance range

anglSubsetApr = era5_sfcApr.t2m.sel(
    latitude=slice(lats[0] + latTol, lats[0] - latTol),
    longitude=slice(lons[0] - lonTol, lons[0] + lonTol),
    time=slice(FocalTimePts[1], FocalTimePts[2])
)

In [5]:
#find corresponding temperature values within some tolerance of the era5 times

# Store average temperatures - March
mar_avg_temps = []
tolerance= timedelta(minutes=30)
anglStation = anglStation.to_dict('records')

#loop through it
for t in anglSubsetMar.time.values:

    focalTime = pd.to_datetime(pd.Timestamp(t))
    # Subset data within the time tolerance
    subset = [
        entry['temperature'] for entry in anglStation
        if abs(datetime.fromisoformat(entry['DATE']) - focalTime) <= tolerance and str(entry['temperature_Quality_Code']) == '5'
    ]

    # Compute the average temperature if any entries matched
    if subset:
        avg = sum(subset) / len(subset)
    else:
        avg = float('nan')
        
    mar_avg_temps.append(avg)
    
#get the difference between these time points and the closest station values
marDiffs = np.array([anglSubsetMar.mean(dim='latitude').values - 272.15]).flatten() - mar_avg_temps
angl_marDiffs = list(filterfalse(isnan, marDiffs))

# Store average temperatures - April
apr_avg_temps = []
tolerance= timedelta(minutes=30)

#loop through it
for t in anglSubsetApr.time.values:

    focalTime = pd.to_datetime(pd.Timestamp(t))
    # Subset data within the time tolerance
    subset = [
        entry['temperature'] for entry in anglStation
        if abs(datetime.fromisoformat(entry['DATE']) - focalTime) <= tolerance and str(entry['temperature_Quality_Code']) == '5'
    ]

    # Compute the average temperature if any entries matched
    if subset:
        avg = sum(subset) / len(subset)
    else:
        avg = float('nan')
        
    apr_avg_temps.append(avg)
    
#get the difference between these time points and the closest station values
aprDiffs = np.array([anglSubsetApr.mean(dim='latitude').values - 272.15]).flatten() - apr_avg_temps
angl_aprDiffs = list(filterfalse(isnan, aprDiffs))

In [6]:
#get some summary statistics
angl_avgDiff = statistics.mean([*angl_marDiffs, *angl_aprDiffs])
angl_medDiff = statistics.median([*angl_marDiffs, *angl_aprDiffs])

In [ ]:
means = [angl_avgDiff, beau_avgDiff, conr_avgDiff, galv_avgDiff, hous_avgDiff]
medians = [angl_medDiff, beau_medDiff, conr_medDiff, galv_medDiff, hous_medDiff]
OutputDF = pd.DataFrame(np.column_stack((means, medians)), columns=["mean", "median"], 
                        index=["Angleton", "Beaumont", "Conroe", "Galveston", "Houston"])
OutputDF.to_csv("../../03ProcessedData/ERA5Errors.csv")